# Classification d'art abstrait par espaces latents


In [ ]:
!pip install umap-learn --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial.distance import cdist

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.manifold import TSNE

try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATA_DIR     = "/kaggle/input/datasets/raphallegrandtp/abstrait-v4/abstrait-v4/"
OUTPUT_DIR   = Path("outputs/")
OUTPUT_DIR.mkdir(exist_ok=True)

IMG_SIZE     = 256
LATENT_DIM   = 128
BETA         = 4.0
BATCH_SIZE   = 32
EPOCHS       = 80
LR           = 3e-4
WEIGHT_DECAY = 1e-5
PATIENCE     = 15
VAL_SPLIT    = 0.15
SEED         = 42
EMBED_DIM    = 64 
EMBED_EPOCHS = 150

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomCrop(IMG_SIZE, padding=16),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_transforms)
CLASS_NAMES  = full_dataset.classes
N_CLASSES    = len(CLASS_NAMES)

n_val   = int(len(full_dataset) * VAL_SPLIT)
n_train = len(full_dataset) - n_val
train_set, val_set = random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

counts = np.zeros(N_CLASSES, dtype=int)
for _, lbl in full_dataset.samples:
    counts[lbl] += 1

## Architecture β-VAE

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.GELU(),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
    def forward(self, x):
        return x + self.block(x)


class ArtBetaVAE(nn.Module):
    def __init__(self, latent_dim=128, img_size=256):
        super().__init__()
        self.latent_dim = latent_dim
        final_spatial = img_size // (2 ** 5)
        flat_dim = 512 * final_spatial ** 2

        self.encoder = nn.Sequential(
            nn.Conv2d(3,   64,  4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),  nn.GELU(), ResidualBlock(64),
            nn.Conv2d(64,  128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(), ResidualBlock(128),
            nn.Conv2d(128, 256, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.GELU(), ResidualBlock(256),
            nn.Conv2d(256, 512, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.GELU(), ResidualBlock(512),
            nn.Conv2d(512, 512, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.Flatten()
        )
        self.fc_mu     = nn.Linear(flat_dim, latent_dim)
        self.fc_logvar = nn.Linear(flat_dim, latent_dim)

        self.decoder_input = nn.Linear(latent_dim, flat_dim)
        self.decoder = nn.Sequential(
            nn.Unflatten(1, (512, final_spatial, final_spatial)),
            nn.ConvTranspose2d(512, 512, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512), nn.GELU(),
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,  64, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,    3, 4, stride=2, padding=1),
            nn.Tanh()
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        if self.training:
            return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)
        return mu

    def decode(self, z):
        return self.decoder(self.decoder_input(z))

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

_vgg = vgg16(weights=VGG16_Weights.DEFAULT).features[:16].to(DEVICE).eval()
for p in _vgg.parameters():
    p.requires_grad = False

_vgg_mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(DEVICE)
_vgg_std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(DEVICE)

def to_vgg_space(x):
    x01 = (x + 1) / 2
    return (x01 - _vgg_mean) / _vgg_std  

def perceptual_loss(recon, target):
    return F.mse_loss(_vgg(to_vgg_space(recon)), _vgg(to_vgg_space(target)))

def beta_vae_loss(recon_x, x, mu, logvar, beta=4.0, perceptual_w=0.5):
    B = x.size(0)
    pixel_loss = F.mse_loss(recon_x, x, reduction='sum') / B
    percep     = perceptual_loss(recon_x, x)
    # On combine les deux échelles : pixel_loss est en 'sum', percep en 'mean'
    recon      = 0.5 * pixel_loss + perceptual_w * percep * B
    kl         = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / B
    return recon + beta * kl, recon.item(), kl.item()

## Entraînement β-VAE

In [ ]:
def train_vae(model, train_loader, val_loader,
              epochs=EPOCHS, beta=BETA, patience=PATIENCE):
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    history   = {"train": [], "val": [], "recon": [], "kl": []}
    best_val, patience_count = float("inf"), 0

    for epoch in range(1, epochs + 1):
        model.train()
        tl = rl = kl_t = 0.0
        for imgs, _ in train_loader:
            imgs = imgs.to(DEVICE)
            optimizer.zero_grad()
            recon, mu, logvar = model(imgs)
            loss, r, k = beta_vae_loss(recon, imgs, mu, logvar, beta)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tl += loss.item(); rl += r; kl_t += k
        scheduler.step()

        model.eval()
        vl = 0.0
        with torch.no_grad():
            for imgs, _ in val_loader:
                imgs = imgs.to(DEVICE)
                recon, mu, logvar = model(imgs)
                loss, _, _ = beta_vae_loss(recon, imgs, mu, logvar, beta)
                vl += loss.item()

        n_t, n_v = len(train_loader), len(val_loader)
        history["train"].append(tl/n_t); history["val"].append(vl/n_v)
        history["recon"].append(rl/n_t); history["kl"].append(kl_t/n_t)

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Train {tl/n_t:.1f} | Val {vl/n_v:.1f} | "
                  f"Recon {rl/n_t:.1f} | KL {kl_t/n_t:.1f} | "
                  f"LR {scheduler.get_last_lr()[0]:.1e}")

        if vl/n_v < best_val:
            best_val, patience_count = vl/n_v, 0
            torch.save(model.state_dict(), OUTPUT_DIR / "best_vae.pth")
        else:
            patience_count += 1
            if patience_count >= patience:
                print(f"Early stopping à l'epoch {epoch}")
                break

    model.load_state_dict(torch.load(OUTPUT_DIR / "best_vae.pth"))
    print(f"Meilleure val : {best_val:.1f}")
    return history

model   = ArtBetaVAE(latent_dim=LATENT_DIM, img_size=IMG_SIZE).to(DEVICE)
history = train_vae(model, train_loader, val_loader)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title, color in [
    (axes[0], None,    "Perte totale",       None),
    (axes[1], "recon", "Reconstruction MSE", "seagreen"),
    (axes[2], "kl",    "KL-Divergence",      "darkorange"),
]:
    if key is None:
        ax.plot(history["train"], label="Train", color="steelblue")
        ax.plot(history["val"],   label="Val",   color="tomato", ls="--")
        ax.legend()
    else:
        ax.plot(history[key], color=color)
    ax.set_title(title); ax.set_xlabel("Epoch")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()

## Extraction des vecteurs latents (en mémoire)

In [ ]:
def extract_latent_vectors(model, data_dir, tf, device=DEVICE):
    ds     = datasets.ImageFolder(root=data_dir, transform=tf)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
    model.eval()
    mus, lbls = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            mu, _ = model.encode(imgs.to(device))
            mus.append(mu.cpu().numpy())
            lbls.append(labels.numpy())
    X = np.concatenate(mus)
    y = np.concatenate(lbls)
    print(f"Vecteurs : {X.shape} | Classes : {len(ds.classes)}")
    return X, y

X_latent, y_labels = extract_latent_vectors(model, DATA_DIR, eval_transforms)

np.save(OUTPUT_DIR / "latent_vectors.npy", X_latent)
np.save(OUTPUT_DIR / "labels.npy",         y_labels)

## Extraction des features CNN (ResNet50 pré-entraîné, figé)

Le CNN ImageNet capture la structure/géométrie que le VAE manque (cf. analyse des plus-proches-voisins).

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

_resnet = resnet50(weights=ResNet50_Weights.DEFAULT)
_resnet.fc = nn.Identity()
_resnet.eval().to(DEVICE)
for p in _resnet.parameters():
    p.requires_grad = False

resnet_transform = ResNet50_Weights.DEFAULT.transforms()

def extract_cnn_features(backbone, data_dir, transform, device=DEVICE):
    ds     = datasets.ImageFolder(root=data_dir, transform=transform)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)
    feats, lbls = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            feats.append(backbone(imgs.to(device)).cpu().numpy())
            lbls.append(labels.numpy())
    return np.concatenate(feats), np.concatenate(lbls)

X_cnn, y_cnn = extract_cnn_features(_resnet, DATA_DIR, resnet_transform)
print(f"Features CNN : {X_cnn.shape}")
assert (y_cnn == y_labels).all(), "Erreur d'ordre"

np.save(OUTPUT_DIR / "cnn_features.npy", X_cnn)

## Fusion VAE + CNN

Le VAE capture surtout la couleur/style global, le CNN capture la structure/géométrie. La fusion combine les deux signaux.

In [ ]:
from sklearn.decomposition import PCA

pca_cnn       = PCA(n_components=128)
X_cnn_reduced = pca_cnn.fit_transform(StandardScaler().fit_transform(X_cnn))

X_vae_s = StandardScaler().fit_transform(X_latent)
X_cnn_s = StandardScaler().fit_transform(X_cnn_reduced)

X_fused = np.concatenate([0.7 * X_vae_s, 1.3 * X_cnn_s], axis=1)

np.save(OUTPUT_DIR / "fused_vectors.npy", X_fused)

## Visualisation de l'espace latent

In [ ]:
TOP_N   = min(20, N_CLASSES)
counts_arr = np.bincount(y_labels, minlength=N_CLASSES)
top_cls = np.argsort(counts_arr)[::-1][:TOP_N]
mask    = np.isin(y_labels, top_cls)
palette = plt.colormaps.get_cmap("tab20")
colors  = {c: palette(i / TOP_N) for i, c in enumerate(top_cls)}

n_cols  = 2 if HAS_UMAP else 1
fig, axes = plt.subplots(1, n_cols, figsize=(8 * n_cols, 7))
if n_cols == 1:
    axes = [axes]

tsne_2d = TSNE(n_components=2, perplexity=30, max_iter=1000,
               random_state=SEED).fit_transform(X_latent[mask])
for c in top_cls:
    m = y_labels[mask] == c
    axes[0].scatter(tsne_2d[m, 0], tsne_2d[m, 1],
                    c=[colors[c]], label=CLASS_NAMES[c], alpha=0.7, s=20)
axes[0].set_title("t-SNE du VAE")
axes[0].legend(fontsize=6, markerscale=2, ncol=2); axes[0].axis("off")

if HAS_UMAP:
    umap_2d = umap.UMAP(n_neighbors=15, min_dist=0.1,
                         random_state=SEED).fit_transform(X_latent[mask])
    for c in top_cls:
        m = y_labels[mask] == c
        axes[1].scatter(umap_2d[m, 0], umap_2d[m, 1],
                        c=[colors[c]], label=CLASS_NAMES[c], alpha=0.7, s=20)
    axes[1].set_title("UMAP du VAE")
    axes[1].legend(fontsize=6, markerscale=2, ncol=2); axes[1].axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "latent_vae.png", dpi=150, bbox_inches="tight")
plt.show()

## Classification par mouvement artistique

On regroupe les ~356 artistes en ~14 mouvements bien alimentés.
Ajustez le mapping selon votre connaissance du dataset.

In [ ]:
MOVEMENT_MAP = {
    "geometric": [
        "albers","herbin","mondrian","vantongerloo","vasarely","malevitch",
        "el-lissitzky","rodtchenko","klucis","popova","rozanova","stazewski",
        "strzeminski","vordemberge-gildewart","gorin","freundlich","domela",
        "carlsund","del-marle","taeuber-arp","huszar","khidekel","nemours",
        "morellet","tomasello","le-parc","soto","cruz-diez","paternosto",
        "reggiani","lussigny","fangor","reutersvard","peire","vezelay",
        "jaray","riley","schoonhoven",
    ],
    "action_painting": [
        "pollock","de-kooning","krasner","mitchell","frankenthaler",
        "hartung","soulages","mathieu","wols","schneider","reigl","atlan",
        "degottex","dmitrienko","marfaing","francis","still","guitet",
        "messagier","singier","bertholle","le-moal","bazaine","manessier",
        "bissiere","tal-coat","esteve","deyrolle","pagava",
        "poliakoff","poliakof",
    ],
    "tachisme_informel": [
        "fautrier","dubuffet","michaux","tapies","millares","burri","vedova",
        "music","saura","appel","alechinsky","jorn","dotremont","bryen",
        "benrath","ubac","loubchansky","masson","noel",
    ],
    "surrealisme": [
        "ernst","miro","matta","dali","masson","toyen","seligmann","sage",
        "tanning","rahon","lamba","dominguez","onslow-ford","varo","maar",
        "bellmer","graverol","papazoff","pailthorpe","colquhoun",
    ],
    "futurisme_orphisme": [
        "balla","boccioni","severini","prampolini","pannaggi","cappa",
        "delaunay-r","delaunay-s","kupka","leger-f",
        "de-souza-cardoso","crotti","villon","redon","serusier",
    ],
    "colour_field": [
        "rothko","newman","kelly","marden","diebenkorn","asse","magnelli",
        "youngerman","stella","de-stael",
    ],
    "nouveau_realisme": [
        "klein","arman","tinguely","saint-phalle","rotella",
        "fontana","manzoni","castellani","scheggi","venet",
    ],
    "street_art": [
        "basquiat","haring","futura-2000","dondi-white","phase-2","ripoulin",
    ],
    "art_brut": [
        "dubuffet","crepin","lesage","stojka","gabori",
        "napangardi","petyarre","aborigene","timms",
    ],
    "expressionnisme": [
        "klee","kandinsky","arp","schwitters","nay","lupertz","kiefer",
        "auerbach","kossof","freundlich","schonberg",
    ],
    "abstraction_lyrique": [
        "de-stael","zao","vieira-da-silva","szenes","helion","patel",
        "khanna","chagall","atlan","masson","lanskoy",
    ],
    "arte_povera_conceptuel": [
        "penone","merz","boetti","boeti","buren","parmentier","toroni",
        "dezeuze","viallat","devade","cane","frize","marclay","long","burroughs",
    ],
    "kinetic_op": [
        "vasarely","le-parc","soto","cruz-diez","morellet","schoffer",
        "schoonhoven","tomasello","fangor","riley","halley","stella",
    ],
    "non_occidental": [
        "el-anatsui","nkanga","damoah","opoku","afro","melehi","mehadji",
        "araeen","shemza","angkasapura","kadiweu","gutai","shiraga","kazuo",
        "ufan","lee","kim","park","paek","min","farmanfarmaian","tanavoli","nasseri",
    ],
}

artist_to_mv = {a: mv for mv, artists in MOVEMENT_MAP.items() for a in artists}

y_mv_str = [artist_to_mv.get(CLASS_NAMES[l], "divers") for l in y_labels]
le_mv    = LabelEncoder()
y_mv     = le_mv.fit_transform(y_mv_str)
MV_NAMES = list(le_mv.classes_)

from collections import Counter
for mv, n in sorted(Counter(y_mv_str).items(), key=lambda x: -x[1]):
    print(f"  {mv:<30} : {n:>4} images")

In [ ]:
mask_known = np.array(y_mv_str) != "divers"
X_mv = X_fused[mask_known]
y_m  = y_mv[mask_known]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_mv, y_m, test_size=0.2, random_state=SEED, stratify=y_m
)

svm = Pipeline([
    ("sc",  StandardScaler()),
    ("clf", SVC(kernel="rbf", C=10, gamma="scale",
                probability=True, random_state=SEED))
])
svm.fit(X_tr, y_tr)

cv_s = cross_val_score(svm, X_mv, y_m,
                        cv=StratifiedKFold(5, shuffle=True, random_state=SEED))
print(f"Accuracy test : {svm.score(X_te, y_te):.3f}")
print(f"CV 5-fold     : {cv_s.mean():.3f} ± {cv_s.std():.3f}")

y_pred_mv = svm.predict(X_te)
mv_present = sorted(np.unique(y_m))
mv_labels  = [MV_NAMES[i] for i in mv_present]

cm = confusion_matrix(y_te, y_pred_mv, normalize="true")
fig, ax = plt.subplots(figsize=(13, 11))
ConfusionMatrixDisplay(cm, display_labels=mv_labels).plot(
    ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
plt.title("Confusion — Mouvements artistiques")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_movements.png", dpi=150, bbox_inches="tight")
plt.show()

## Espace siamois

In [ ]:
def build_prototypes(X, y, names):
    return {names[i]: X[y == i].mean(axis=0)
            for i in np.unique(y) if (y == i).sum() > 0}

def proto_predict(queries, prototypes, metric="cosine"):
    names  = list(prototypes.keys())
    matrix = np.stack([prototypes[n] for n in names])
    dists  = cdist(queries, matrix, metric=metric)
    return [names[i] for i in np.argmin(dists, axis=1)], dists

counts_arr = np.bincount(y_labels, minlength=N_CLASSES)
multi_cls  = np.where(counts_arr >= 2)[0]

y_true_loo, y_pred_loo = [], []
for cls in multi_cls:
    for idx in np.where(y_labels == cls)[0]:
        mask_tr = np.ones(len(X_fused), bool)
        mask_tr[idx] = False
        protos = build_prototypes(X_fused[mask_tr], y_labels[mask_tr], CLASS_NAMES)
        pred, _ = proto_predict(X_fused[idx:idx+1], protos)
        y_true_loo.append(CLASS_NAMES[cls])
        y_pred_loo.append(pred[0])

acc_proto = sum(t == p for t, p in zip(y_true_loo, y_pred_loo)) / len(y_true_loo)
print(f"Prototypical LOO accuracy (artistes ≥ 2 images) : {acc_proto:.3f}")
print(f"Nombre d'artistes évalués : {len(multi_cls)}")

### Réseau Siamois (Triplet Loss)

Plutôt qu'une perte contrastive (paires), on utilise une **Triplet Loss** : pour chaque ancre, on tire un exemple positif (même artiste) et un négatif (artiste différent), et on force `distance(ancre, positif) < distance(ancre, négatif) - margin`. Ce signal relatif est plus riche qu'une simple paire et apprend mieux la hiérarchie de similarité, ce qui devrait resserrer les clusters comme le Cluster 1 observé (actuellement trop hétérogène).

In [5]:
class TripletDataset(Dataset):
    """
    Génère des triplets (ancre, positif, négatif).
    - ancre / positif : même artiste (2 images différentes)
    - négatif         : artiste différent
    Pour les artistes à 1 seule image (pas de positif possible), l'ancre est
    réutilisée comme positif (force le réseau à au moins être stable sur la même image,
    et le négatif continue d'éloigner les autres artistes).
    """
    def __init__(self, X, y, n_triplets=50000):
        self.X, self.y     = X, y
        self.n_triplets     = n_triplets
        self.cls_idx        = {c: np.where(y == c)[0] for c in np.unique(y)}
        self.classes         = list(self.cls_idx.keys())
        self.multi_cls       = [c for c, idx in self.cls_idx.items() if len(idx) >= 2]

    def __len__(self):
        return self.n_triplets

    def __getitem__(self, _):
        anchor_cls = np.random.choice(self.classes)
        idxs = self.cls_idx[anchor_cls]

        if len(idxs) >= 2:
            i_anchor, i_pos = np.random.choice(idxs, 2, replace=False)
        else:
            i_anchor = i_pos = idxs[0]  # singleton : positif = ancre elle-même

        neg_cls = np.random.choice(self.classes)
        while neg_cls == anchor_cls:
            neg_cls = np.random.choice(self.classes)
        i_neg = np.random.choice(self.cls_idx[neg_cls])

        a = torch.tensor(self.X[i_anchor], dtype=torch.float32)
        p = torch.tensor(self.X[i_pos],    dtype=torch.float32)
        n = torch.tensor(self.X[i_neg],    dtype=torch.float32)
        return a, p, n


class EmbeddingNet(nn.Module):
    def __init__(self, in_dim=128, out_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, 128),    nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, out_dim),
        )
    def forward(self, z):
        return F.normalize(self.net(z), p=2, dim=1)


# ─── ENTRAÎNEMENT ─────────────────────────────────────────────────────────────
LATENT_DIM  = X_fused.shape[1]   # 256 (VAE 128 + CNN réduit 128)
MARGIN      = 0.3

triplet_ds     = TripletDataset(X_fused, y_labels, n_triplets=50000)
triplet_loader = DataLoader(triplet_ds, batch_size=256, shuffle=True, num_workers=2)

embed_net   = EmbeddingNet(LATENT_DIM, EMBED_DIM).to(DEVICE)
opt_s       = optim.AdamW(embed_net.parameters(), lr=1e-3, weight_decay=1e-4)
sched_s     = optim.lr_scheduler.CosineAnnealingLR(opt_s, T_max=EMBED_EPOCHS)
triplet_crit = nn.TripletMarginLoss(margin=MARGIN, p=2)

print("Entraînement réseau siamois (Triplet Loss)...")
history_triplet = []
for epoch in range(1, EMBED_EPOCHS + 1):
    embed_net.train()
    total, n_active = 0.0, 0
    for a, p, n in triplet_loader:
        a, p, n = a.to(DEVICE), p.to(DEVICE), n.to(DEVICE)
        ea, ep, en = embed_net(a), embed_net(p), embed_net(n)
        loss = triplet_crit(ea, ep, en)
        opt_s.zero_grad(); loss.backward(); opt_s.step()
        total += loss.item()

        # Suivi du % de triplets "actifs" (loss > 0), utile pour diagnostiquer
        with torch.no_grad():
            d_pos = F.pairwise_distance(ea, ep)
            d_neg = F.pairwise_distance(ea, en)
            n_active += (d_pos + MARGIN > d_neg).sum().item()

    sched_s.step()
    avg_loss = total / len(triplet_loader)
    pct_active = 100 * n_active / (len(triplet_loader) * triplet_loader.batch_size)
    history_triplet.append(avg_loss)

    if epoch % 10 == 0:
        print(f"  Epoch {epoch:3d}/{EMBED_EPOCHS} | Loss {avg_loss:.4f} | "
              f"Triplets actifs {pct_active:.1f}%")

torch.save(embed_net.state_dict(), OUTPUT_DIR / "siamese_embedder_triplet.pth")

plt.figure(figsize=(8, 4))
plt.plot(history_triplet, color="steelblue")
plt.title("Triplet Loss — convergence")
plt.xlabel("Epoch"); plt.ylabel("Loss moyenne")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "triplet_loss_curve.png", dpi=150)
plt.show()

NameError: name 'Dataset' is not defined

In [ ]:
# Extraction des embeddings dans l'espace siamois (entraîné par Triplet Loss)
embed_net.eval()
with torch.no_grad():
    X_embedded = embed_net(
        torch.tensor(X_fused, dtype=torch.float32).to(DEVICE)
    ).cpu().numpy()


# LOO dans l'espace siamois
y_true_s, y_pred_s = [], []
for cls in multi_cls:
    for idx in np.where(y_labels == cls)[0]:
        mask_tr = np.ones(len(X_embedded), bool)
        mask_tr[idx] = False
        protos = build_prototypes(X_embedded[mask_tr], y_labels[mask_tr], CLASS_NAMES)
        pred, _ = proto_predict(X_embedded[idx:idx+1], protos)
        y_true_s.append(CLASS_NAMES[cls])
        y_pred_s.append(pred[0])

acc_siam = sum(t == p for t, p in zip(y_true_s, y_pred_s)) / len(y_true_s)


## Comparaison des espaces latents

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, X_vis, title in [
    (axes[0], X_fused,    "Espace VAE+CNN fusionné"),
    (axes[1], X_embedded, "Espace Siamois"),
]:
    X_s  = X_vis[mask]
    y_s  = y_labels[mask]
    emb  = TSNE(n_components=2, perplexity=30, max_iter=1000,
                random_state=SEED).fit_transform(X_s)
    for c in top_cls:
        m = y_s == c
        ax.scatter(emb[m, 0], emb[m, 1], c=[colors[c]],
                   label=CLASS_NAMES[c], alpha=0.8, s=40,
                   edgecolors="white", linewidths=0.3)
    ax.set_title(f"{title} (t-SNE, Top-20)", fontsize=13, fontweight="bold")
    ax.legend(fontsize=6, markerscale=1.5, ncol=2, loc="lower right")
    ax.axis("off")

plt.suptitle("Un meilleur clustering = espace plus discriminant", fontsize=14)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "comparison_spaces.png", dpi=150, bbox_inches="tight")
plt.show()

## Diagnostic : séparation proche/éloigné

Vérifie numériquement si la Triplet Loss a bien resserré les clusters fragiles identifiés visuellement (ex: Cluster 1, hétérogène).

In [ ]:
from sklearn.metrics import silhouette_samples
from sklearn.cluster import KMeans

# Reclustering rapide sur le nouvel espace siamois pour comparer au clustering précédent
N_CLUSTERS = 25
km = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10)
cluster_labels_new = km.fit_predict(X_embedded)

sil = silhouette_samples(X_embedded, cluster_labels_new)

print(f"{'Cluster':<10}{'N images':<12}{'Silhouette moyen':<20}")
for c in range(N_CLUSTERS):
    mask = cluster_labels_new == c
    if mask.sum() > 0:
        print(f"{c:<10}{mask.sum():<12}{sil[mask].mean():<20.3f}")

print(f"\nSilhouette global : {sil.mean():.3f}")
print("→ Comparer ce score à celui obtenu sur X_fused avant Triplet Loss "
      "pour valider l'amélioration.")

## Inférence sur une nouvelle image

In [ ]:
def predict_artist(image_path, vae, resnet, embedder,
                   pca_cnn, scaler_vae, scaler_cnn,
                   X_db, y_db, names, top_k=5, device=DEVICE):
    """Pipeline complet : VAE + CNN + fusion + embedding siamois."""
    from PIL import Image

    img = Image.open(image_path).convert("RGB")

    # μ du VAE
    tensor_vae = eval_transforms(img).unsqueeze(0).to(device)
    vae.eval()
    with torch.no_grad():
        mu, _ = vae.encode(tensor_vae)
    mu = mu.cpu().numpy()

    # Features CNN
    tensor_cnn = resnet_transform(img).unsqueeze(0).to(device)
    resnet.eval()
    with torch.no_grad():
        feat_cnn = resnet(tensor_cnn).cpu().numpy()

    # Fusion identique à l'entraînement
    mu_s      = scaler_vae.transform(mu)
    feat_pca  = pca_cnn.transform(StandardScaler().fit(X_cnn).transform(feat_cnn))
    feat_s    = scaler_cnn.transform(feat_pca)
    fused     = np.concatenate([0.7 * mu_s, 1.3 * feat_s], axis=1)

    # Embedding siamois
    embedder.eval()
    with torch.no_grad():
        emb = embedder(torch.tensor(fused, dtype=torch.float32).to(device)).cpu().numpy()

    protos  = build_prototypes(X_db, y_db, names)
    pnames  = list(protos.keys())
    pmatrix = np.stack([protos[n] for n in pnames])
    dists   = cdist(emb, pmatrix, metric="cosine")[0]
    top_idx = np.argsort(dists)[:top_k]

    print(f"Top-{top_k} artistes les plus proches :")
    for rank, idx in enumerate(top_idx, 1):
        sim = 1 - dists[idx]
        print(f"  {rank}. {pnames[idx]:<25} {'█'*int(sim*30)} {sim:.3f}")
    return pnames[top_idx[0]]

# Exemple :
# predict_artist("mon_image.jpg", model, _resnet, embed_net,
#                 pca_cnn, StandardScaler().fit(X_latent), StandardScaler().fit(X_cnn_reduced),
#                 X_embedded, y_labels, CLASS_NAMES)

In [2]:
embed_net.eval()
with torch.no_grad():
    X_embedded = embed_net(torch.tensor(X_fused, dtype=torch.float32).to(DEVICE)).cpu().numpy()
np.save("/kaggle/working/embedded_vectors_triplet.npy", X_embedded)
print(f"Sauvegardé : {X_embedded.shape}")

NameError: name 'embed_net' is not defined